# Night Shift: get the recycling line moving again
**One notebook · five labs · 16 short checkpoints · CPU only**

The lights have changed, one inspection camera is noisy, the belt-speed sensor is down,
and the sorter needs a new routing rule. You have one shift to recover the inspection pipeline.

**Your deliverable:** corrected images → surface inspection → motion estimate → stitched belt map
→ a trained three-class sorter → a two-route adapted model → an attention audit.

This is a **generated teaching simulator**, not photographs or a physically validated robot.
Objects are geometric proxies for cardboard tiles, caps and cans. Main data, labels and lighting
are created locally; no model weights, videos or datasets are downloaded. A final optional cell
can open one existing course photograph to expose the domain gap.

| Station | You build | Original lab | Suggested active time |
|---|---|---|---|
| [1. Fix the inspection booth](#booth) | Colour, intrinsic layers, normals, height | Lab1 | 30–40 min |
| [2. Clean the signal](#signal) | Gaussian derivatives, gradients, LoG/DoG | Lab2 | 25–35 min |
| [3. Recover belt motion](#motion) | Features, Lucas–Kanade; RAFT reading | Lab3 | 20–30 min |
| [4. Build a common map](#map) | SIFT matches, affine RANSAC, warping, blending | Lab4 | 30–40 min |
| [5. Teach and audit the sorter](#sorter) | CNN, training, transfer, Grad-CAM | Lab5 | 35–45 min |

Times are estimates, not a deadline. Skip the optional ablations on a first pass.
No installation essays, video codecs, long training runs or repeated plotting boilerplate.

## Start here
Launch instructions and coverage details are in `README.md` beside this notebook. Save a copy as
`my-master-lab.ipynb` before working. **Edit only the exercise cells and your short answers.**
Run a challenge's check cell after editing. `TODO` or `CHECK FAILED` is not a pass; it does not
prevent you trying another station. Later stations start from clearly labelled supplied captures,
not silently inserted solutions. The final replay uses your implementations and stays locked
until all 16 code checkpoints pass. Restart and Run All before trusting the final score.

Allowed: NumPy, the named SciPy operations, PyTorch and the supplied plotting/interpolation helpers.
Implement the requested algorithm; don't replace it with a high-level solver that bypasses it.
Mathematical `lstsq` is allowed. Reference answers are **separate** in `solutions.py` and
`SOLUTIONS.md`; this notebook never imports them.

**Conventions:** RGB float `[0,1]`, HWC arrays; gray images HW; tensor images NCHW.
Arrays use `I[y,x]`; coordinates and flows are `(x,y)` and `(u,v)` with y down.
The height graph uses `n ∝ [-p,-q,1]`, hence `p=-nx/nz`, `q=-ny/nz`.
Lab1's printed positive ratios conflict with HC2a; do not mix the two conventions.

In [ ]:
from pathlib import Path
import copy
import sys
import numpy as np
import matplotlib.pyplot as plt
import cv2
from scipy import ndimage, signal
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

get_ipython().run_line_magic('matplotlib', 'inline')
candidates = [Path.cwd(), Path.cwd() / 'study-pack' / 'master-lab']
LAB_DIR = next((p for p in candidates if (p / 'mission_support.py').is_file()), None)
if LAB_DIR is None:
    raise RuntimeError('Open from the course root or study-pack/master-lab; see README.md.')
sys.path.insert(0, str(LAB_DIR.resolve()))
import mission_support as support
from mission_support import sample_bilinear
mission = support.Mission()
packet = mission.packet
print(f'CPU | NumPy {np.__version__} | OpenCV {cv2.__version__} | PyTorch {torch.__version__}')
print('No course files are changed. All main captures are generated locally.')
support.gallery([packet['raw'], packet['view_a'], packet['view_b']],
                ['Colour-cast inspection feed', 'Belt camera A', 'Belt camera B'])

<a id="booth"></a>
## 1 — Fix the inspection booth
The surface-inspection booth stops the belt and takes seven fixed-view images under calibrated lights.
The later motion/camera-pair captures are **different acquisitions**, not moving photometric-stereo inputs.

### 01 · The magenta mystery [Lab1 colour/constancy]

#### Why colour needs more than one representation

The camera measures light reflected by the material **and** modified by the illuminant. A channel gain
can therefore change the apparent colour without changing the object. We will compare representations
that answer different questions, then apply one simple colour-constancy assumption.

For a pixel with channels R,G,B, lightness is
$$\ell=\frac{\max(R,G,B)+\min(R,G,B)}{2}.$$
This is not the weighted grayscale/luma conversion used elsewhere. Normalized RGB instead records
relative channel proportions:
$$r=\frac{R}{R+G+B},\quad g=\frac{G}{R+G+B},\quad b=\frac{B}{R+G+B}.$$
A common positive brightness multiplier cancels, but different channel gains do not. At a black pixel
the denominator is zero; the task defines the output as zero rather than inventing a colour.
Opponent channels compare red with green, yellow with blue, and total brightness. Unlike RGB,
the first two opponent channels can legitimately be negative.

Grey-World assumes the **average material colour in the view is neutral**. Let
$$\mu_c=\frac{1}{HW}\sum_{y,x}I_{yxc},\qquad
\bar\mu=\frac{\mu_R+\mu_G+\mu_B}{3},\qquad
I'_{yxc}=I_{yxc}\frac{\bar\mu}{\mu_c}.$$
This estimates three global gains, not a different correction for every pixel. A neutral strip lets
you inspect the result, but this algorithm does not locate or explicitly use that strip.

**Translate to arrays.** Start with `(H,W,3)`. A per-pixel channel reduction removes the last axis;
a spatial channel mean reduces axes `(0,1)`. Retain singleton dimensions when a denominator or gain
must broadcast back over the image. For example, `(H,W,1)` broadcasts per pixel, whereas `(1,1,3)`
broadcasts per channel. Convert to float before arithmetic. Do not stretch negative opponent values
or clip corrected RGB until preparing a display: that would change the quantities you are computing.

#### Your implementation

Write `colour_tools(rgb)`, returning a dict:
- `lightness`: `(max(R,G,B)+min(R,G,B))/2`, shape HW.
- `nrgb`: each pixel divided by its channel sum; black pixels stay zero.
- `opponent`: `(R-G)/sqrt(2)`, `(R+G-2B)/sqrt(6)`, `(R+G+B)/sqrt(3)`, stacked HWC.
- `balanced`: Grey-World correction; spatial channel means to their common mean. Zero-mean channels stay zero.

Work in float, **do not clip or quantize intermediate results**. Clip only for display if necessary.
The calibration strip is neutral gray in the generator.
**Your call:** would Grey-World still be a good assumption if the whole view were brown cardboard?
<details><summary>Hint</summary>

Channel means use axes `(0,1)`, per-pixel channel sums use `-1`.
`keepdims=True` keeps broadcasting honest. Use `np.divide(..., where=...)` for zeros.

</details>

In [ ]:
def colour_tools(rgb):
    # Return dict(lightness=..., nrgb=..., opponent=..., balanced=...).
    raise NotImplementedError('01: channel axes and colour correction')

In [ ]:
colour = mission.run('01', colour_tools)
if colour is not None:
    support.gallery([packet['raw'], np.clip(colour['balanced'],0,1), colour['nrgb']],
                    ['Before', 'Your Grey-World correction', 'Your normalized RGB'])
    support.colour_views(np.clip(colour['balanced'],0,1))

### 02 · Repaint the material, not the shadows [Lab1 intrinsic images]

#### Separate what the object is from how it is lit

An intrinsic-image model writes a linear RGB observation as
$$I(x,y,c)=A(x,y,c)S(x,y).$$
Here A is material reflectance (albedo), and S is a scalar shading field. The simplified model assumes
the same shading factor affects all three channels. It ignores coloured interreflection and specular
highlights. In real images, recovering A and S from I alone is ambiguous; **this task supplies them**.
You are implementing reconstruction and a material edit, not solving intrinsic decomposition.

Painting the displayed image directly would replace shadowed pixels and bright pixels alike, making
the result look flat. Instead, change the material and render it again under the original shading:
$$A'(x,y,:)=\begin{cases}t,&M(x,y)=1,\\A(x,y,:),&M(x,y)=0,\end{cases}
\qquad I'=A'S.$$
The target t is an RGB albedo, not the final observed RGB. A red material can still look dark in shadow.

**Translate to arrays.** Albedo has shape `(H,W,3)`, shading and mask have shape `(H,W)`, and the
target has shape `(3,)`. Boolean selection with the mask selects complete RGB pixels. Make a copy
before replacement so later experiments still have the original albedo. Multiplication needs shading
with shape `(H,W,1)`; otherwise NumPy tries to align W with the colour axis. The mask defines which
material changes; a zero pixel is not a substitute for a mask. Outside the mask, reconstruct the
original material under its shading rather than copying an unshaded albedo value to the output.

#### Your implementation

The simulator supplies material RGB `albedo`, gray `shading` and an explicit object `mask`.
Implement `recolour`: copy albedo, replace it by `target` **only inside the mask**, then reconstruct
with `I=A*S` (broadcast shading to HWC). Do not infer a mask from black pixels.
**Predict first:** if shading doubles, what happens to every nonzero reconstructed channel?

In [ ]:
def recolour(albedo, shading, mask, target):
    raise NotImplementedError('02: preserve shading and outside-mask material')

In [ ]:
recoloured = mission.run('02', recolour)
if recoloured is not None:
    support.gallery([packet['albedo']*packet['shading'][...,None], packet['shading'], recoloured],
                    ['Original rendering', 'Unchanged shading', 'Your recoloured object'])

### 03 · Is that can dented? [Lab1 photometric stereo]

#### Recover geometry by changing the lights, not the viewpoint

The booth holds the object and camera fixed. Under a diffuse Lambertian model, one pixel observed
under light j satisfies
$$i_j=\rho\,\mathbf l_j^T\mathbf n=\mathbf l_j^T\mathbf g,
\qquad \mathbf g=\rho\mathbf n.$$
The light vector includes the calibrated light direction/intensity convention; the supplied vectors
are normalized. Stacking m exposures gives
$$\underbrace{\mathbf i}_{m\times1}=
\underbrace{L}_{m\times3}\underbrace{\mathbf g}_{3\times1},\qquad
\hat{\mathbf g}=\arg\min_{\mathbf g}\|L\mathbf g-\mathbf i\|_2^2.$$
Three independent light directions are needed to constrain three unknown components. More exposures
make an overdetermined fit, not a guarantee against shadows or highlights. Prefer a least-squares
solver to explicitly forming and inverting the normal equations.

**Translate to arrays.** Reshape `(m,H,W)` into `(m,P)`, where `P=H*W`. Each column now represents
one pixel across all exposures. A multi-right-hand-side least-squares solve returns `(3,P)`.
Recover albedo and normals by
$$\rho_p=\sqrt{g_{1p}^2+g_{2p}^2+g_{3p}^2},\qquad
\mathbf n_p=\mathbf g_p/\rho_p.$$
Reduce the component axis, not the pixel axis. Transpose the normal array to `(P,3)` **before**
reshaping it to `(H,W,3)`; a direct reshape of `(3,P)` mixes channels and pixels. Zero albedo means
no observable normal, and the defined zero output is a validity convention, not a physical direction.

**Why the weighted version is different.** For each pixel, form `D=diag(i)` and solve
$$\min_{\mathbf g}\|DL\mathbf g-D\mathbf i\|_2^2
=\min_{\mathbf g}\sum_j i_j^2(\mathbf l_j^T\mathbf g-i_j)^2.$$
Thus each row of L is multiplied by its corresponding intensity, and the right-hand side contains
intensity squared. A length-m vector must gain a column axis to scale rows of an `(m,3)` array.
Do not use one pixel's D for the entire image: its shadow pattern varies spatially. Zero observations
drop out, but unusually bright specular observations receive **more** influence. The comparison table
tests this limitation rather than assuming weighting is always better.

#### Your implementation

`images` is `(m,H,W)`, lights `L` is `(m,3)`. Solve `i=L@g`, `g=rho*n` per pixel.
Return `(rho, normals)` of shapes HW and HWC. For zero rho return a zero normal, not NaN.
Unweighted mode can solve all pixels at once with `lstsq`.

Also implement Lab1's shadow weighting: solve `diag(i) @ L @ g = diag(i) @ i` per pixel.
It minimizes residuals with weights **i²**, not i. Loops over pixels are fine for weighted mode.
This rig uses linear radiometry and rank-3 lights; valid clean dot products are positive.
**Your call:** the table compares shadow and specular corruption. Why can weighting help one and hurt the other?
For RGB, repeat the fit per channel. A zero-reflectance channel cannot supply a normal direction:
mask it or use a valid shared-geometry estimate, rather than divide by zero.
<details><summary>Hint</summary>

Reshape observations to `(m,H*W)`. Normalize g along its 3-vector axis,
not by one norm of the entire image. Extra images cannot fix rank-deficient lighting.

</details>

In [ ]:
def photometric(images, lights, weighted=False):
    raise NotImplementedError('03: least squares, albedo, unit normals and weights')

In [ ]:
surface = mission.run('03', photometric)
if surface is not None:
    rho, normals = surface
    support.gallery([rho, (normals+1)/2, packet['shadow'][1]],
                    ['Your albedo', 'Your normals encoded as RGB', 'One damaged exposure'])
    support.photo_summary(packet, photometric)
    rgb_burst = packet['burst'][...,None]*np.array([.7,.4,0.])
    rgb_fits = [photometric(rgb_burst[...,c],packet['lights']) for c in range(3)]
    print('RGB channel mean albedos (blue is deliberately black):',[round(r.mean(),3) for r,n in rgb_fits])
    print('Blue-channel normal is invalid, even though other channels observe the surface.')

### 04 · Normals are not a height map [Lab1 integration]

#### Surface orientation tells you a slope, not an absolute height

Represent the surface as a height graph z(x,y), with slopes p and q. Its tangent vectors are
`(1,0,p)` and `(0,1,q)`, whose cross product gives a normal proportional to `(-p,-q,1)`:
$$\mathbf n=\frac{(-p,-q,1)}{\sqrt{1+p^2+q^2}},\qquad
p=\frac{\partial z}{\partial x}=-\frac{n_x}{n_z},\quad
q=\frac{\partial z}{\partial y}=-\frac{n_y}{n_z}.$$
This explains the signs; do not choose them by whether a plot looks like a bump. Our y coordinate
increases down the array, and the surface convention is declared consistently with that coordinate.

Integration adds local height increments along a path. For horizontal spacing Δx and vertical
spacing Δy, an increment is approximately pΔx or qΔy. This exercise sets both spacings to one.
To reach `(y,x)`, first move across row zero, then down column x:
$$z(y,x)=\sum_{a=0}^{x-1}p(0,a)+\sum_{b=0}^{y-1}q(b,x).$$
Empty sums are zero, which fixes the top-left anchor. A cumulative sum must therefore be shifted
into positions starting at index one; including the current pixel's outgoing slope is an off-by-one error.

**Translate to arrays.** Extract the three normal components from the final axis. p and q are HW
arrays. Fill the first row from horizontal increments, then propagate each later row from the row
above using q. This is a deliberately simple path integrator, not a global least-squares reconstruction.
Compute the diagnostic
$$C=\frac{\partial p}{\partial y}-\frac{\partial q}{\partial x}.$$
In NumPy, the y derivative is axis 0 and x derivative is axis 1. For a smooth, integrable field C
should be near zero. Noise can make different paths disagree; discrete differentiation/integration
also introduces approximation error. Adding any constant to z leaves the normals unchanged.

#### Your implementation

Use `p=-nx/nz`, `q=-ny/nz`, unit grid spacing, anchor `z[0,0]=0`.
Integrate **across the top row**, then **down each column** using outgoing/left differences:
`z[0,x+1]=z[0,x]+p[0,x]`; `z[y+1,:]=z[y,:]+q[y,:]`.
Return `dict(height=z, curl=dp/dy-dq/dx)`. Use `np.gradient(..., axis=...)` for the curl diagnostic.
The supplied normals have nonzero nz; a real vertical/invalid normal needs masking.
Central-difference slopes and this one-sided sum use different discretizations: even clean height is approximate.
**Predict first:** a plane with p=1,q=-.5 rises by how much over dx=4,dy=2?
**Your call:** why can the opposite integration path disagree on a noisy field? Which height ambiguity remains?

In [ ]:
def integrate(normals):
    raise NotImplementedError('04: slopes, path and mixed derivatives')

In [ ]:
relief = mission.run('04', integrate)
if relief is not None:
    support.gallery([packet['height']-packet['height'][0,0], relief['height'], relief['curl']],
                    ['Generated height, same anchor', 'Your path integral', 'Integrability residual'])

<a id="signal"></a>
## 2 — Clean the signal, keep the edges
A supplied noisy belt capture lets you continue even if the booth is unfinished. You will build the
filters, not write another plotting framework. Noise-removal comparisons here are a short supporting
experiment; the original lab's long optional noise/PSNR/Gabor sweeps are omitted.

### 05 · Build one useful filter family [Lab2 Gaussian construction]

#### Smooth nearby pixels without moving the whole image

A Gaussian uses large weights near the centre and smaller weights farther away. Sigma controls its
spread in pixels; radius controls where we truncate its infinite support. They are different parameters.
On integer positions `-r,...,r`, construct
$$\tilde g(x)=e^{-x^2/(2\sigma^2)},\qquad
g(x)=\frac{\tilde g(x)}{\sum_{a=-r}^{r}\tilde g(a)}.$$
Normalize the **actual sampled vector**, rather than assuming the continuous normalization constant
makes a truncated discrete sum exactly one. A unit-sum smoothing kernel preserves a constant image.

Differentiate the Gaussian to obtain filters that respond to change:
$$g'(x)=-\frac{x}{\sigma^2}g(x),\qquad
g''(x)=\left(\frac{x^2}{\sigma^4}-\frac1{\sigma^2}\right)g(x).$$
The first derivative is odd; the second is even. These are sampled analytic derivatives using the
normalized g above, not finite differences of the vector. An odd derivative should have zero sum
up to roundoff. The truncated second derivative can have a small nonzero sum. Neither should be
normalized to unit sum: doing so destroys its meaning or divides by nearly zero.

**Translate to arrays.** x, g, dg and ddg are all length `K=2*radius+1`; powers and products are
elementwise. Forming `outer(g,g)` produces a `(K,K)` kernel. A 2D isotropic Gaussian separates:
$$G(x,y)=g(x)g(y).$$
One horizontal pass and one vertical pass therefore reproduce the 2D filtering with matching boundary
rules, at roughly 2K rather than K² multiply-adds per output pixel. The provided comparison checks
that claim numerically. Symmetric reflection limits artificial borders; it does not make image edges
equivalent to observations outside the camera. When applying **odd** derivative kernels later,
remember that convolution flips a kernel whereas cross-correlation does not.

#### Your implementation

For `x=-radius..radius`, sample `g=exp(-x**2/(2*sigma**2))` and normalize its actual sum.
Return `(g, dg, ddg)` with `dg=-(x/sigma**2)*g`, `ddg=(x**2/sigma**4-1/sigma**2)*g`.
These derivatives use the sampled normalized g; do **not** normalize derivatives to sum 1.
**Your call:** why do two 1D passes equal an outer-product 2D filter? Compare K² and 2K work/pixel.

In [ ]:
def gaussian1d(sigma=1.2, radius=4):
    raise NotImplementedError('05: normalized Gaussian and its derivatives')

In [ ]:
kernels = mission.run('05', gaussian1d)
if kernels is not None:
    g, dg, ddg = kernels
    image = packet['noisy']
    smooth = ndimage.correlate1d(ndimage.correlate1d(image,g,axis=0),g,axis=1)
    direct = signal.correlate2d(image,np.outer(g,g),mode='same',boundary='symm')
    print('Largest direct/separable difference:', np.max(np.abs(smooth-direct)))
    support.gallery([packet['clean'], image, smooth, ndimage.median_filter(image,3)],
                    ['Clean teaching target', 'Noisy', 'Your Gaussian', 'Supplied median comparison'], columns=4)
    plt.figure(figsize=(7,2)); plt.plot(g,label='g'); plt.plot(dg,label="g′"); plt.plot(ddg,label="g″")
    plt.legend(); plt.show()

### 06 · How strong is the edge? [Lab2 gradients]

#### Make the direction and strength of an edge measurable

An image gradient consists of two directional derivatives:
$$\nabla I=(I_x,I_y),\qquad
\|\nabla I\|=\sqrt{I_x^2+I_y^2},\qquad
\theta=\operatorname{atan2}(I_y,I_x).$$
The gradient points toward increasing intensity, perpendicular to a local intensity contour. It is
not the direction along the edge. An orientation is not meaningful where both derivatives are zero,
even though a numeric implementation may return zero there.

Our horizontal Sobel operator combines a left/right difference with vertical smoothing:
$$S_x=\frac18\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix},\qquad S_y=S_x^T.$$
The factor 1/8 makes the interior response to a unit x-ramp equal to one. Here we use **correlation**:
the kernel is used in the written order, so brighter pixels on the right give positive gx. A convolution
with the same written kernel reverses that sign.

**Translate to arrays.** Cast the HW input to float before filtering: unsigned arithmetic cannot
represent negative derivatives. Correlate with each 3×3 kernel using `mode='same'` and
`boundary='symm'`; both results stay HW. Magnitude and angle then use elementwise operations.
An unrestricted matrix norm would collapse the whole image to one number. `atan(gy/gx)` both
divides by zero and loses quadrant information; `atan2` accepts the two arrays directly.
Its argument order is **y component first, x component second**. Keep signed derivatives for later
calculations. Scaling each derivative independently for display would change the recovered angles.

#### Your implementation

Return `(gx, gy, magnitude, angle_radians)`. Use **cross-correlation** with
`sx=[[-1,0,1],[-2,0,2],[-1,0,1]]/8`, `sy=sx.T`, same output, symmetric boundaries.
`signal.correlate2d` is allowed; a built-in Sobel/gradient routine is not the implementation here.
Cast to float first. Magnitude is `sqrt(gx**2+gy**2)` **per pixel**; angle is `atan2(gy,gx)`.
Do not min-max normalize derivatives used in calculations; normalize only visualizations.
**Predict:** on `I=x+2y`, what are gx,gy away from the boundary? Is gradient direction along or across an edge?

In [ ]:
def gradients(image):
    raise NotImplementedError('06: signed derivatives, magnitude and quadrant-safe angle')

In [ ]:
edges = mission.run('06', gradients)
if edges is not None:
    gx, gy, magnitude, angle = edges
    support.gallery([gx,gy,magnitude,angle],['gx','gy','Gradient magnitude','Angle (radians)'],columns=4)

### 07 · Same edge, three routes [Lab2 LoG/DoG]

#### Find changes at a chosen spatial scale

The Laplacian adds second derivatives in x and y. It responds to rapid spatial changes, but applying
it directly to a noisy image also amplifies noise. Gaussian smoothing selects a scale before measuring
curvature. The continuous identity is
$$\nabla^2(G_\sigma*I)=(\nabla^2G_\sigma)*I.$$
This motivates the first two routes, but their discrete constructions differ. Route one uses a sampled
Gaussian followed by a small finite-difference Laplacian. Route two samples the analytic LoG:
$$\nabla^2G_\sigma(x,y)=
\frac{x^2+y^2-2\sigma^2}{\sigma^4}G_\sigma(x,y).$$
Its centre is negative with this convention, surrounded by positive weights. A constant image should
have zero second derivative. Truncation can leave the sampled kernel with nonzero sum, so subtract
its **mean** to remove that unwanted DC response. This is not unit-sum normalization.

The third route subtracts two smoothing scales:
$$D=G_{k\sigma}*I-G_\sigma*I,\qquad k>1.$$
Broad structures survive both smoothings and largely cancel; very fine structures are suppressed by
both. Intermediate scales remain. For nearby scales,
$$D\approx(k-1)\sigma^2(\nabla^2G_\sigma)*I.$$
Thus matching the location of structures is different from matching their numerical amplitudes.

**Translate to arrays.** Create x and y coordinate grids of shape `(K,K)` on the same support as
`outer(g,g)`, not image-sized grids. Apply every filter with the same output size and boundary mode.
For DoG, construct and normalize a new Gaussian at k*sigma; multiplying the narrow kernel's
values by k does not make it wider. Keep the subtraction signed. Taking an absolute value too early
removes polarity and hides zero crossings. The three maps should resemble related scale-selective
responses, but finite sampling, truncation and different discrete Laplacians prevent exact equality.

#### Your implementation

Return `dict(sequential=..., direct=..., dog=...)`, all HW, same symmetric boundaries.
Reuse your `gaussian1d`; use `signal.correlate2d`, not a built-in LoG.
1. Gaussian smooth, then correlate with `[[0,1,0],[1,-4,1],[0,1,0]]`.
2. Make `G=outer(g,g)`, then sampled LoG `((x**2+y**2-2*sigma**2)/sigma**4)*G`.
   Subtract the sampled kernel's mean to correct its finite-support DC leakage; correlate directly.
3. Smooth at sigma and k*sigma on the same support, then **wide minus narrow**.
**Your call:** which is low-/high-/band-pass? Why aren't these three sampled responses exactly equal?
For k close to 1, divide DoG by `(k-1)*sigma**2` to compare with LoG; don't compare unscaled amplitudes.

In [ ]:
def frequency_maps(image, sigma=1.2, k=1.6, radius=4):
    raise NotImplementedError('07: sequential/direct LoG and two-scale DoG')

In [ ]:
frequency = mission.run('07', frequency_maps)
if frequency is not None:
    support.gallery(list(frequency.values()), list(frequency.keys()))

<a id="motion"></a>
## 3 — The speed sensor is down
Two supplied gray frames move the same belt by a small displacement. Derivatives for the LK
equations are supplied from smoothed frames so filtering mistakes do not block this station.
Station 2 still builds and checks your own gradient implementation.

### 08 · Pick a point that can actually be tracked [Lab3 features]

#### A useful feature constrains motion in two directions

An edge can slide along itself without much appearance change. A corner or textured patch has
intensity variation in more than one direction. Summarize that evidence in a local second-moment matrix:
$$M=\sum_{(x,y)\in W}\begin{bmatrix}I_x^2&I_xI_y\\I_xI_y&I_y^2\end{bmatrix}
=\begin{bmatrix}a&b\\b&c\end{bmatrix}.$$
This is a 2×2 matrix **at every image location**, not one covariance matrix for the entire picture.
Its eigenvalues measure gradient evidence along two principal directions. Both small means flat;
one small means an edge; both large means a promising trackable patch.

Shi–Tomasi uses the smaller eigenvalue, whereas Harris avoids an explicit eigenvalue decomposition:
$$\lambda_{\min}=\frac{a+c-\sqrt{(a-c)^2+4b^2}}2,\qquad
R=ac-b^2-k(a+c)^2.$$
Harris scores can be negative at edges; that is expected. Positive definiteness comes from gradient
information, not from replacing every negative score by zero.

**Translate to arrays.** First form the three HW product images, **then** aggregate each spatially.
Squaring an averaged gradient is not the same operation as averaging squared gradients: opposing
gradients could cancel before their evidence is counted. `uniform_filter` computes a mean, so multiply
by the window area to obtain the sums required here. The window parameter is a side length, making
the area `window**2`. Combining the resulting a,b,c arrays elementwise yields full-image score maps.
The helper performs thresholding, local-maximum selection and border exclusion afterwards. Your
function should return scores, not an already-thresholded list of coordinates.

#### Your implementation

Form local **sums** `a=sum(gx²), b=sum(gx*gy), c=sum(gy²)` in a window.
Allowed aggregation: `ndimage.uniform_filter(...,window,mode='reflect')*window**2`.
Return `dict(shi=lambda_min, harris=det(M)-k*trace(M)**2)` over the full image,
where `M=[[a,b],[b,c]]` and `lambda_min=(a+c-sqrt((a-c)**2+4*b**2))/2`.
**Predict:** flat region, straight edge, textured corner—which has two useful eigenvalues?
This compresses the original optional feature warm-up into a dependency for mandatory LK.

In [ ]:
def structure_scores(gx, gy, window=5, k=.04):
    raise NotImplementedError('08: second moments and corner scores')

In [ ]:
corners = mission.run('08', structure_scores)
if corners is not None:
    support.gallery([packet['frame0'],corners['shi'],corners['harris']],
                    ['Choose reliable points', 'Shi–Tomasi', 'Harris'])

### 09 · Recover belt speed [Lab3 Lucas–Kanade]

#### Turn brightness constancy into a solvable local system

If a visible point moves by (u,v) between nearby frames without changing brightness,
$$I(x+u,y+v,t+1)\approx I(x,y,t).$$
A first-order Taylor expansion gives the optical-flow constraint
$$I_xu+I_yv+I_t=0.$$
One pixel provides one equation for two unknowns: this is the aperture problem. Lucas–Kanade adds
the assumption that all pixels in a small patch share the same flow. For P pixels, stack
$$A=\begin{bmatrix}I_x(1)&I_y(1)\\\vdots&\vdots\\I_x(P)&I_y(P)\end{bmatrix},
\quad b=-\begin{bmatrix}I_t(1)\\\vdots\\I_t(P)\end{bmatrix},
\quad \hat{\mathbf v}=\arg\min_{\mathbf v}\|A\mathbf v-b\|^2.$$
The solution has just two components, regardless of patch size. The matrix `A.T@A` is the same
gradient-product structure studied in checkpoint 08. Its smaller eigenvalue tells us whether there
is enough evidence for motion in **both** directions. A numerical least-squares solver can return a
minimum-norm vector for a rank-deficient edge, but that does not make the missing component observable.

**Translate to arrays.** Flatten gx,gy,it in the same order so each row describes the same pixel.
Stack gx and gy as **columns**, producing `(P,2)`, not `(2,P)`. Negate it because the supplied temporal
difference is second frame minus first frame. Test conditioning before solving; return the specified
NaNs when the patch cannot support a two-dimensional estimate. The result is `(u,v)`, not `(row,column)`.

The small displacement is essential to the linear approximation. Repeated warping or image pyramids
can extend the useful range, but this task implements one local solve. A brightness jump breaks the
model even when motion has not changed. The helper's long arrows are display scaling only; never
multiply the returned flow by that factor when converting pixels/frame into physical speed.

#### Your implementation

Assume constant local flow. Flatten the patch; A has rows `[gx,gy]`, b=`-it`.
If `eigvalsh(A.T@A)[0] < min_eigenvalue`, return `[nan,nan]`: reject unobservable motion.
Otherwise return least-squares `(u,v)` in **pixels/frame**, no visualization scale factor.
`it` is second-minus-first intensity, not the other sign.
**Your call:** larger windows add equations, but what can go wrong at a motion boundary?
A quarter-resolution `(u,v)=(2,-1)` is what displacement at original resolution?

In [ ]:
def lk_patch(gx, gy, it, min_eigenvalue=1e-5):
    raise NotImplementedError('09: observable 2D flow, correct temporal sign')

In [ ]:
motion = mission.run('09', lk_patch)
if motion is not None and corners is not None:
    tracked_motion = support.tracks(packet, corners, lk_patch)
    # A deliberate lighting failure; no geometry has changed.
    region = np.s_[38:55,38:55]
    print('Same patch with a brightness jump:',
          lk_patch(packet['gx'][region],packet['gy'][region],packet['it'][region]+.1))

### RAFT pit stop — five lab questions in one model-reading challenge [Lab3 Q17–21]
**No RAFT weights or inference are run here.** This retains the taught concepts without the
original video decoding/download setup. RAFT = Recurrent All-Pairs Field Transforms.

Three parts: (1) learned feature/context encoders; (2) all-pairs feature correlations with a pooled
**correlation pyramid**; (3) recurrent updates that repeatedly refine one flow estimate.
The flow stays at a fixed working grid (commonly 1/8 input resolution) and is upsampled for output;
it does **not** mean the feature extractor never downsamples. Unlike coarse-to-fine LK, the
correlation pyramid pools matching evidence, not a succession of resized flow fields.

Read the essential API shape, not executable setup:
```python
model = raft_small(weights=weights).eval()
with torch.no_grad():
    flow_iterations = model(frame1_batch, frame2_batch)
final_flow = flow_iterations[-1]  # N,2,H,W; components u,v in input-pixel units
display = flow_to_image(final_flow)  # visualization, NOT class labels
```
**Write three short answers below:**
1. Why can the same ball have different flow colours in two frame pairs? Which quantities do colour/direction
   and intensity/magnitude represent, and why must you inspect the visualization scale?
2. How do correlation lookup and recurrent updates help both small/large displacements without discarding
   a small feature at a coarse flow level? Name the three components in your explanation.
3. Lighting change broke your LK estimate. Does a trained network guarantee the right answer under
   occlusion or unfamiliar lighting? Distinguish learned robustness from a valid visible correspondence.

In [ ]:
raft_answers = {
    'colours_and_units': 'YOUR ANSWER',
    'architecture_and_correlation_pyramid': 'YOUR ANSWER',
    'failure_and_domain_shift': 'YOUR ANSWER',
}

<a id="map"></a>
## 4 — Two cameras, one belt map
You get overlapping views with a small affine change. OpenCV supplies **only SIFT detection/descriptors**.
You supply matching, robust fitting, inverse coordinate mapping and mask-aware blending.
This rig is affine by construction; a general perspective panorama would need a homography.

### 10 · Find evidence, not lookalikes [Lab4 SIFT matching]

#### A descriptor is local evidence, not a unique identity

SIFT detects stable locations across scale, assigns a dominant orientation, and describes a normalized
neighbourhood using gradient histograms. Its usual descriptor contains 4×4 spatial cells with eight
orientation bins each: 128 values. The provided extractor does this work. You implement the decision
about which descriptors correspond across cameras.

For descriptors a_i and b_j, use Euclidean distance
$$d_{ij}=\|a_i-b_j\|_2.$$
For each a_i, let d₁ and d₂ be its smallest and second-smallest distances to **different candidates**
in camera B. Keep the nearest neighbour only when `d1 < threshold*d2`. This tests distinctiveness:
a small best distance is less convincing if another candidate is almost equally good. It does not
enforce mutual matching or geometric agreement; those are separate tests.

**Translate to arrays.** For N source and M target descriptors, the distance table is `(N,M)`.
Search/sort along its target axis, independently for each source row. You need both target indices
and distance values: the returned pair is `(source_index, nearest_target_index)`, not two distances.
Avoid taking square roots by using
$$\|a-b\|^2=\|a\|^2+\|b\|^2-2a^Tb,$$
but then the retention test is `d1_squared < threshold**2*d2_squared`. Floating-point subtraction
can produce tiny negative squared distances; clamp those to zero before interpreting them. Equal
zero-distance alternatives remain ambiguous and must be rejected. Return an integer `(0,2)` array
when no matches survive, so downstream column indexing still has a defined shape.

#### Your implementation

`a` and `b` are descriptor matrices N×D and M×D. Return integer `(N_kept,2)` index pairs.
Compute the closest and second closest **L2 distances** per row of a; retain if `d1/d2 < threshold`.
Reject d2=0 ties; if fewer than 2 candidates, return an empty `(0,2)` array.
Using squared distances? Compare their ratio to **threshold²**.
**Your call:** can a ratio test prove the two points belong to the same physical location?
<details><summary>Hint</summary>

Squared distance can be formed without an N×M×D tensor:
`sum(a*a)[:,None] + sum(b*b)[None,:] - 2*a@b.T`. Clamp tiny negative roundoff to zero.

</details>

In [ ]:
def ratio_matches(a, b, threshold=.75):
    raise NotImplementedError('10: distances, ambiguity and index pairs')

In [ ]:
matching = mission.run('10', ratio_matches)
if matching is not None:
    features, pairs = matching
    print(len(pairs),'ratio-test matches. Descriptor width:',features['desc_a'].shape[1])
    support.show_matches(packet,features,pairs)

### 11 · Ignore the convincing wrong matches [Lab4 affine RANSAC]

#### Separate transformation estimation from outlier rejection

An affine map can translate, rotate, scale and shear a plane:
$$\begin{bmatrix}x'\\y'\end{bmatrix}=
\underbrace{\begin{bmatrix}a&b&t_x\\c&d&t_y\end{bmatrix}}_{M}
\begin{bmatrix}x\\y\\1\end{bmatrix}.$$
There are six unknowns; each matched pair supplies two equations. With source points as N×2 rows,
append ones to form X of shape N×3. Solve `X@C ≈ dst` for the two right-hand sides together;
C is 3×2, so the requested transform M is its transpose. Three source points must be noncollinear
for X to have rank three. More pairs allow a least-squares fit, but ordinary LS gives outliers influence.

RANSAC treats matches as competing evidence. For each trial: sample three distinct pairs, fit a
candidate, predict **all** target points, and measure target-plane residuals
$$r_i=\|M[x_i,y_i,1]^T-[x'_i,y'_i]^T\|_2.$$
An inlier has residual at most the threshold in pixels. Keep the largest consensus, then fit again
using every pair in that consensus. The final LS fit uses more evidence than the winning minimal
sample. Recompute the reported mask against that final model. Degenerate samples should be skipped,
not terminate the whole search; failure to find any usable consensus should be reported.

**Translate to arrays.** Use one index sample to select corresponding rows in both arrays. Generate
predictions with the same source→target convention as your fitter, and reduce the two-coordinate
residual along axis 1. Keep the boolean mask separately from its integer count. Seed the generator
once outside the loop; resetting it each trial would repeat the same sample. If the inlier fraction is w
and sample size s, an idealized all-inlier sample has probability w^s; for desired success p,
$$T\geq\frac{\log(1-p)}{\log(1-w^s)}.$$
This estimate assumes independent trials and usable geometry; the fixed trial budget is not certainty.

#### Your implementation

`fit_affine(src,dst)` returns M of shape 2×3, with `dst ≈ [src,1] @ M.T`.
Fit by LS; reject rank<3 (collinear source points). At least three point pairs are needed.
`ransac_affine` uses a seeded RNG, samples 3 distinct pairs, rejects degeneracy, counts residuals
`<=threshold` in target pixels, keeps the largest consensus, then **refits all its inliers**.
Return `(matrix, boolean_inlier_mask)`; reject fewer than 3 input pairs/no usable consensus.
Use your `fit_affine`, not OpenCV's RANSAC implementation.
**Predict:** which needs more trials: w=.8 or w=.4, same sample size and success probability?
<details><summary>Hint</summary>

Use `np.random.default_rng(seed)`, `choice(...,replace=False)`.
An all-inlier sample still needs nondegenerate geometry. A finite budget is not a certainty guarantee.

</details>

In [ ]:
def fit_affine(src, dst):
    raise NotImplementedError('11a: affine least squares')

def ransac_affine(src, dst, threshold=2., trials=200, seed=8):
    raise NotImplementedError('11b: sample, score, retain, refit')

In [ ]:
alignment = mission.run('11', (fit_affine,ransac_affine))
if alignment is not None and matching is not None:
    matrix,inliers = alignment
    print('Inliers:',inliers.sum(),'/',len(pairs),'| Your A→B affine:')
    print(matrix)
    support.show_matches(packet,features,pairs,inliers)

### 12 · Build the map without holes or black-pixel bugs [Lab4 warping/stitching]

#### Estimate geometry first, then resample appearance

An affine transform maps continuous coordinates, but images store samples on integer grids. Forward
mapping source pixels and rounding their destinations can leave holes or place several samples at
one location. Inverse warping instead visits every destination pixel and asks where it came from:
$$\begin{bmatrix}x_s\\y_s\\1\end{bmatrix}=
\begin{bmatrix}M_{00}&M_{01}&M_{02}\\M_{10}&M_{11}&M_{12}\\0&0&1\end{bmatrix}^{-1}
\begin{bmatrix}x_d\\y_d\\1\end{bmatrix}.$$
The forward matrix remains source→destination; only the **sampling query** uses its inverse.
For an affine transform, the last homogeneous coordinate stays one. A projective transform would
also require division by that coordinate, but is not the model used here.

**Translate to arrays.** `np.indices((H,W))` returns y then x. Flatten those grids, stack x,y,ones
as a 3×(H*W) coordinate matrix, transform it, then reshape source x and y to the output HW shape.
Pass those arrays to the supplied sampler in **x,y order**. It uses bilinear interpolation: the four
neighbouring source samples contribute according to fractional horizontal/vertical position. It also
returns a geometric validity mask. Out-of-bounds zeros are padding; in-bounds black pixels are real data.

After both cameras occupy the same canvas, feathering combines them with nonnegative coverage weights:
$$I=\frac{w_AI_A+w_BI_B}{w_A+w_B}.$$
The supplied distance-to-boundary weights favour view interiors. In a region covered only by A,
normalization returns A unchanged. Where neither view contributes, define zero explicitly rather than
divide by zero. For RGB, give HW weights a singleton channel axis so the **same weight** applies to all
channels. Feathering smooths intensity seams; it cannot correct a wrong transform or parallax.

#### Your implementation

Implement two short functions:
- `inverse_warp(image,matrix,output_shape)`: output_shape is `(H,W)`. Matrix maps **source→destination**.
  Build the destination grid, apply the inverse 3×3 homogeneous matrix, then call the supplied
  `sample_bilinear(image,xs,ys)`; it returns `(warped,valid_mask)`. You compute the coordinates.
- `feather_blend(a,b,wa,wb)`: return `(a*wa+b*wb)/(wa+wb)` with channel broadcasting and zero
  where both weights are zero. Weights are coverage-aware, not `image != 0`.

The helper supplies corner bounds/canvas translation and distance-to-boundary weights.
**Your call:** why does forward pixel splatting leave holes? Why is real black not a reliable missing-data marker?

In [ ]:
def inverse_warp(image, matrix, output_shape):
    raise NotImplementedError('12a: backward coordinates and valid coverage')

def feather_blend(a, b, wa, wb):
    raise NotImplementedError('12b: weighted combination, including valid black')

In [ ]:
mapped = mission.run('12', (inverse_warp,feather_blend))
if mapped is not None and alignment is not None and matching is not None:
    belt_map = support.panorama(packet,alignment[0],inverse_warp,feather_blend)

<a id="sorter"></a>
## 5 — Teach, adapt, and audit the sorter
You are building a **classifier, not an object detector**: object crops are provided.
Training/validation/test contain independent generated object instances. Colours vary independently
of labels; shapes still make this much easier than real material recognition. No MNIST/CIFAR/ImageNet download.
Use validation for choices. Open the held-out test score only in the final replay.

In [ ]:
loaders = support.make_loaders()
sample_x,sample_y = next(iter(loaders['train']))
support.gallery([a.permute(1,2,0).numpy() for a in sample_x[:6]],
                [support.LABELS[int(y)] for y in sample_y[:6]],columns=6,size=1.7)
print('Batch:',sample_x.shape,'| train/val/test:',[len(loaders[k].dataset) for k in ['train','val','test']])

### 13 · Small network, real tensor arithmetic [Lab5 CNN mechanics]

#### Learn useful features instead of hand-designing the last decision

The earlier stations use specified filters and geometric models. The sorter instead learns convolution
weights from labelled examples. Each convolution shares its weights across image positions, ReLU
introduces a nonlinearity, and pooling reduces spatial resolution. This task classifies already-provided
object crops; it does not learn to locate objects on a whole belt.

For one spatial axis, a convolution's output size is
$$H_{out}=\left\lfloor\frac{H_{in}+2P-D(K-1)-1}{S}\right\rfloor+1.$$
Here dilation D=1, kernel K=3, stride S=1 and padding P=1 preserve height/width. A 2×2 max pool
with default stride two halves them. Keep a shape trace on paper: batch size N stays unchanged,
convolutions change channel count, and pooling changes spatial dimensions.

Global average pooling converts the final feature maps A into one number per feature channel:
$$f_{nk}=\frac1{H_fW_f}\sum_{y,x}A_{nkyx},\qquad z_n=Wf_n+b.$$
The three entries of z are logits: unnormalized class scores. The next task's cross-entropy handles
their conversion into log probabilities; a softmax layer here would apply the wrong input contract.

**Translate to tensors.** Input is `(N,3,32,32)`, not HWC. Put the two convolution/ReLU/pool blocks
in `self.features`, and the final linear layer in `self.classifier` so transfer and CAM can access them.
Average only spatial dimensions `(2,3)`, retaining batch and feature axes. Do not flatten all 8×8
positions into the classifier: that defines a different architecture and parameter count. Learnable
parameter counts are `(K*K*Cin+1)*Cout` for each biased convolution and `(Din+1)*Dout` for the
linear layer. Pooling, ReLU and spatial averaging add no learned parameters in this architecture.

#### Your implementation

Build `Sorter` with attributes `features` and `classifier`:
`Conv(3→8,3×3,pad1) → ReLU → MaxPool(2) → Conv(8→16,3×3,pad1) → ReLU → MaxPool(2)`.
Then **global spatial mean** (N,16,8,8→N,16), then `Linear(16,3)`. Include ordinary biases.
Return **logits**, not softmax; cross-entropy includes log-softmax.
Use ordinary PyTorch layers, not an imported pretrained network.
**Predict before running:** trainable parameters? Feature-map shape after each stage? Does stride change weights?
<details><summary>Hint</summary>

Conv params `(Kh*Kw*Cin+1)*Cout`; FC `(Din+1)*Dout`.
GAP, ReLU and pooling have no ordinary learned weights. Forward must use the two named attributes.

</details>

In [ ]:
class Sorter(nn.Module):
    def __init__(self):
        super().__init__()
        raise NotImplementedError('13: features and classifier')

    def forward(self, x):
        raise NotImplementedError('13: spatial mean and logits')

In [ ]:
model = mission.run('13', Sorter)
if model is not None:
    print(model)
    print('Trainable parameters:',sum(p.numel() for p in model.parameters() if p.requires_grad))

### 14 · The training loop is part of the model [Lab5 learning/evaluation]

#### Optimizing weights and measuring performance are separate operations

For logits z and correct class y, multiclass cross-entropy is
$$\mathcal L(z,y)=-\log\frac{e^{z_y}}{\sum_c e^{z_c}}
=-z_y+\log\sum_c e^{z_c}.$$
Use PyTorch's stable implementation rather than manually exponentiating scores. Targets are integer
class indices of shape `(N,)`, not one-hot RGB masks. Training computes gradients of the mean batch
loss and lets the optimizer update the parameters.

**Implement the sequence deliberately.** Set training mode first. Clear previous parameter gradients;
PyTorch accumulates them by default. Run the model, compute scalar loss, call backward to populate
gradients, then call the optimizer step to change weights. Backward alone does not train the model.
Detach the returned reporting value so history does not retain computation graphs. Do not detach
the logits or loss **before** backward, because that disconnects the learning path.

Evaluation has two separate controls: `eval()` selects inference behaviour for layers such as dropout
and BatchNorm, while `no_grad()` disables recording derivative computations. Neither replaces the
other. Predict each sample by the largest logit along the class axis. There is no need to compute
softmax first because it preserves score ordering.

**Aggregate by examples, not by batches.** If batch b has n_b examples and mean loss L_b,
$$L_{dataset}=\frac{\sum_b n_bL_b}{\sum_b n_b},\qquad
accuracy=\frac{\sum_b correct_b}{\sum_b n_b}.$$
Alternatively accumulate sum-reduced batch losses. Averaging batch means directly overweights a
short final batch. Confusion entry `(i,j)` counts true class i predicted as j; increment every example,
including repeated pairs. Derive the number of classes from the logit width, since the later routing
head has two outputs rather than three. Use validation to make choices; reserve the test split for
the final run, not repeated learning-rate selection.

#### Your implementation

`train_step`: train mode → clear gradients → logits → CE → backward → optimizer.step;
return detached **mean batch loss** as a Python float. Clearing gradients must work even if stale gradients exist.
`evaluate`: eval mode **and** no_grad; return dict `accuracy`, `loss` (dataset mean CE), and integer
`confusion` (rows true, columns predicted). Use example-weighted totals, not a mean of batch accuracies.
Derive class count from output width; this must also work on the two-route model later.
Batches are CPU tensors; no device boilerplate is needed.
**Your call:** a smaller last batch exposes what averaging bug? Does `no_grad` turn off dropout?

In [ ]:
def train_step(model, optimizer, x, y):
    raise NotImplementedError('14a: complete training update')

def evaluate(model, loader):
    raise NotImplementedError('14b: modes, sample weights and confusion counts')

In [ ]:
learning = mission.run('14', (train_step,evaluate))
if learning is not None and model is not None:
    history = support.fit(model,loaders,train_step,evaluate,epochs=24)
    print('VALIDATION confusion (rows=true):')
    print(evaluate(model,loaders['val'])['confusion'])

### Optional 3-minute failure lab — change one thing, not five
Same architecture, object instances, batch order and twenty-epoch budget. Rate/optimizer comparisons
share initial weights; initialization comparisons deliberately change them.
Compare Adam, tiny learning rate, SGD, zero/Gaussian/He convolution initialization. These are small training
demonstrations, **not universal optimizer rankings**. A fixed epoch budget is not equal wall-clock cost.
Inspect training loss as well as validation: is the problem underfit, overfit or optimization?
Zero-initialized ReLU branches can fail while the output bias still updates.

**Diagnosis card:** He/ReLU weight variance is about `2/fan_in`; Xavier/Glorot about
`2/(fan_in+fan_out)` under their usual assumptions. Repeated derivative magnitudes below 1 can vanish,
above 1 can explode. Initialization, non-saturating activations, normalization and residual paths
help; clipping limits exploding gradients, not vanished ones. A deeper model with higher **training**
error can have optimization trouble, not overfitting. High train/low validation performance suggests
checking capacity, augmentation, regularization and data leakage. Augmentation must preserve labels.

In [ ]:
RUN_ABLATIONS = False  # switch to True after checkpoint 14
if RUN_ABLATIONS and learning is not None and model is not None:
    comparisons = support.compare_training(Sorter,train_step,evaluate)

### 15 · New job, same backbone [Lab5 transfer learning]

#### Reuse a representation while changing what its outputs mean

The trained sorter can be viewed as two functions:
$$z=h_\phi(f_\theta(x)).$$
The feature extractor f has learned spatial responses to this small source task. The head h maps
those responses to the original three classes. The new routing task has two meanings, so the old
head is not reusable simply by keeping its first two output columns. Replace it with a new two-output
head and learn that mapping using route-labelled data.

Freezing the backbone means optimizing only the new head parameters φ, while preserving θ. The
inputs now also have a changed channel gain, providing a small domain shift. Whether frozen features
remain useful is an empirical validation question, not a guarantee of transfer learning. Large real
domain shifts can require fine-tuning some feature layers and more representative data.

**Translate to model operations.** Deep-copy the whole source before changing anything. An assignment
like `adapted=source` creates an alias, so replacing its head would also destroy the three-class sorter.
Disable gradients for the copied feature parameters, then install a newly initialized linear head with
the same 16 input features and two outputs. Its weights and biases must remain trainable. The existing
training driver builds an optimizer from parameters that require gradients.

Freezing, evaluation mode and no-grad do different jobs. A frozen feature extractor can still run in
training mode; dropout/BatchNorm behaviour would matter in a more complex backbone. Conversely,
wrapping the **entire** adapted forward pass in no-grad would prevent learning the new head. This
particular backbone has no dropout or BatchNorm, but keep the distinctions clear for other networks.
Only the head's `16*2+2` parameters should update; frozen weights still occupy memory and compute.

#### Your implementation

New output meaning: **0=bypass caps, 1=press cardboard/cans**, under a different channel gain.
`transfer_head(source)` must deep-copy the trained model, freeze **features only**, replace the
classifier with `Linear(16,2)` (new head trainable), and return the adapted model.
Never reset the feature weights or mutate your three-class source.
This is **local toy pretraining/domain adaptation**, not an ImageNet/VGG experiment.
Split membership stays by object ID across domains: no test object's recoloured sibling enters training.
**Predict:** how many parameters now update? Explain freezing versus eval mode versus no_grad.

In [ ]:
def transfer_head(source):
    raise NotImplementedError('15: reuse learned weights, change output semantics')

In [ ]:
router = mission.run('15', transfer_head)
if router is not None and learning is not None:
    routes = support.make_loaders(domain='route')
    route_history = support.fit(router,routes,train_step,evaluate,epochs=30)
    print('New-head trainable parameters:',sum(p.numel() for p in router.parameters() if p.requires_grad))
    print('ROUTE VALIDATION confusion:')
    print(evaluate(router,routes['val'])['confusion'])

### 16 · Is it looking at the object? [Lab5 Grad-CAM]

#### Ask which feature locations support a class score

A correct classification does not reveal whether the model used the intended object or a background
shortcut. Grad-CAM builds a coarse class-specific map from feature activations and their influence
on a selected class **logit**. It is a diagnostic, not a segmentation target or proof of causality.

For activation channel k at spatial location (i,j), and selected score s_c, define
$$\alpha_k^c=\frac1{H_fW_f}\sum_{i,j}\frac{\partial s_c}{\partial A_{kij}},
\qquad L^c_{ij}=\operatorname{ReLU}\left(\sum_k\alpha_k^c A_{kij}\right).$$
First average gradients spatially to measure each channel's contribution. Then combine the **spatial
activation maps** using those channel weights. ReLU keeps positive support under this method's
convention. Do not average away the activation locations themselves: that would leave no map to inspect.

**Translate to tensors.** Both inputs have shape `(1,K,Hf,Wf)`. Average gradient axes `(2,3)`, keeping
them as singleton dimensions, to obtain `(1,K,1,1)`. Broadcasting those weights over activations,
then summing channel axis 1, gives `(1,Hf,Wf)`. Select the one batch element to return HW. These
operations stay in PyTorch until you detach and convert the final map to NumPy. The helper already
computes real class-score gradients for you; your task is their correct spatial/channel reduction.

For display, normalize this single map by
$$\hat L=\frac{L-\min L}{\max L-\min L}.$$
If the denominator is zero, return zeros rather than NaNs. This normalization expresses **relative
strength within one map**; it does not make magnitudes comparable across classes or images. The map
is at feature resolution, so it can look blocky beside the input. An empty positive-support map is
possible and does not automatically mean your code is broken. Evaluation mode may be enabled while
autograd remains active: explanations need gradients even when we are not updating weights.

#### Your implementation

The helper computes actual activations A and class-score gradients from your trained model.
`cam_map(features,gradients)` receives NCHW **PyTorch tensors**, N=1.
Average gradients over H,W → channel weights; weighted sum of A over channels → ReLU → HW NumPy.
Min-max scale this one map to `[0,1]`; return zeros if constant.
**Your call:** would a belt-background highlight be a warning? Does a plausible heatmap prove causal reasoning?
Do not run a gradient-based explanation inside no_grad. Eval and gradient recording are separate controls.

In [ ]:
def cam_map(features, gradients):
    raise NotImplementedError('16: gradient weights, channel sum and positive map')

In [ ]:
attention = mission.run('16', cam_map)
if attention is not None and model is not None:
    probe = sample_x[:1]
    activations,gradient_values,target = support.capture_cam(model,probe)
    heatmap = cam_map(activations,gradient_values)
    support.gallery([probe[0].permute(1,2,0).numpy(),heatmap],
                    ['Validation not used here: one TRAIN probe',f'Grad-CAM for {support.LABELS[target]}'],columns=2)

## Final replay — the line must work as a system
This is not a sequence of reference answers: the replay calls your colour correction, shadow-weighted
photometric stereo, height integration, Gaussian/gradient/LoG routines, feature score, LK, ratio test,
affine RANSAC, inverse warp, feathering, trained sorter, evaluation and CAM functions.
Fixed-view shape inspection and moving-belt inspection remain physically separate branches.
Belt crops are supplied at known positions; they are **not independent test examples** or detections.
The untouched generated test split is evaluated here, after your validation-based choices.

In [ ]:
mission.status()
final_report = support.final_replay(
    mission, colour_tools, photometric, integrate, gaussian1d, gradients, frequency_maps,
    structure_scores, lk_patch, ratio_matches, ransac_affine, inverse_warp, feather_blend,
    model, evaluate, cam_map,
)
if final_report is not None and router is not None:
    print('Final ROUTE test accuracy:',round(evaluate(router,support.make_loaders('route')['test'])['accuracy'],3))

### Your shift report — five decisions, not five essays
Write one or two sentences each. Code checks cannot assess these explanations.
1. **Lighting/shape:** where did shadow weighting help, where did it fail, and what height reference is missing?
2. **Signal:** which filter would you deploy and what relevant detail did it remove?
3. **Motion:** which assumptions failed under the brightness jump; would larger windows or RAFT guarantee a repair?
4. **Map:** which matches were rejected, why use inverse warping, and what can't blending fix?
5. **Sorter:** what do validation/confusion/CAM show, what remains untested, and would you ship or hold this system?
Include a real-world test you would require before deployment. Toy accuracy is not that test.

In [ ]:
shift_report = {
    'lighting_and_shape': 'YOUR ANSWER',
    'signal': 'YOUR ANSWER',
    'motion': 'YOUR ANSWER',
    'map': 'YOUR ANSWER',
    'sorter_and_ship_or_hold': 'YOUR ANSWER',
}

### Optional 2-minute reality check — one local course photograph
If you have the original Lab5 image folder, inspect one metal photo with your edge pipeline.
This does not download anything or evaluate a material classifier. Our toy shape labels do **not**
match the real six-material label set. Do not present the toy classifier's output as a material prediction.
**Your call:** which gaps—texture, pose, clutter, lighting, labels, data size—make a real deployment harder?

In [ ]:
RUN_REAL_PHOTO = False
if RUN_REAL_PHOTO and edges is not None:
    photo_dir = LAB_DIR.resolve().parents[1] / 'labs/cv1-lab-5/docs/garbage-classification-dataset/garbage-classification-images/metal'
    photos = sorted(photo_dir.glob('*.jpg'))
    if photos:
        real_rgb = cv2.cvtColor(cv2.imread(str(photos[0])),cv2.COLOR_BGR2RGB)/255.
        real_mag = gradients(support.gray(real_rgb))[2]
        support.gallery([real_rgb,real_mag],['Provided Lab5 photograph','Your gradients on real appearance'],columns=2)
    else:
        print('Local course photograph not found; the core notebook needs no course data.')

**Done.** If you can write the small functions and defend the decisions, you have rehearsed the
main computational ideas of Labs 1–5 without repeating their setup or large experiment grids.
See `README.md` for the exact coverage/omission map and `SOLUTIONS.md` after your attempt.
This is original revision material, not an official replacement for assessed lab submissions.